# NB06 — SLURM and HPC Cluster Workflow

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

Your laptop has 8 cores. A SLURM cluster at EMBL or the Sanger Institute has tens of thousands. When your RNA-seq analysis of 200 samples runs in 2 hours locally, the same pipeline on a cluster — properly parallelized via SLURM job arrays — takes 5 minutes.

SLURM is the most common HPC job scheduler in academic bioinformatics. Knowing how to write a `#SBATCH` script, submit a job array, monitor queue status, and link jobs into a pipeline with Snakemake is a hard prerequisite for most research computing environments.

---

## 2. Intuition

SLURM is like a restaurant with many tables (compute nodes) and many customers (jobs). You place your order (submit a job) with the waiter (scheduler). The scheduler decides when your table is ready based on priority, available resources, and fairness policies. Your script tells SLURM: how many tables (nodes), how many chairs per table (cores), how much space on the table (memory), and how long you'll stay (time limit).

**Job arrays** are the same job script run independently on different inputs — like ordering the same meal for 100 people. SLURM handles parallelism automatically; you write the script for one sample.

**Snakemake** sits on top of SLURM: it generates the job scripts for you and tracks dependencies between pipeline steps. If alignment must finish before counting, Snakemake enforces that order automatically.

---

## 3. Biological background

A typical RNA-seq pipeline has these steps, each suitable for HPC parallelism:

| Step | Tool | CPU | Memory | Time |
|------|------|-----|--------|------|
| Quality trimming | Trimmomatic | 4 | 4 GB | 30 min |
| Alignment | HISAT2/STAR | 8 | 32 GB | 1–2 hr |
| Counting | HTSeq-count | 1 | 8 GB | 1 hr |
| Differential expression | DESeq2 (R) | 1 | 16 GB | 30 min |

With 200 samples and no HPC: 200 × 3.5 hr = 700 hr (29 days). With SLURM job array (50 concurrent): 3.5 hr × 4 batches = 14 hr.

---

## 4. Mathematical explanation

**SLURM resource model:**
- Each **node** has $C$ cores and $M$ GB memory
- Your job requests $c$ cores and $m$ GB from one or more nodes
- SLURM schedules you when $c$ cores and $m$ GB are simultaneously free
- Time limit $T$ is a hard wall: if your job runs over, SLURM kills it

**Throughput with job arrays:**
- $N$ samples, each taking time $t$ on $c$ cores
- With $K$ concurrent array tasks: total wall time $\approx \lceil N/K \rceil \times t$
- Setting `--array=1-200%50` runs 200 tasks, max 50 at a time

---

## 5. Computational explanation

### SLURM directives (#SBATCH)

```bash
#!/bin/bash
#SBATCH --job-name=hisat2_align       # name visible in squeue
#SBATCH --output=logs/%j.out          # stdout (%j = job ID)
#SBATCH --error=logs/%j.err           # stderr
#SBATCH --nodes=1                     # number of nodes
#SBATCH --ntasks=1                    # number of MPI tasks (usually 1 for Python/R jobs)
#SBATCH --cpus-per-task=8             # cores (threads) per task
#SBATCH --mem=32G                     # total memory
#SBATCH --time=02:00:00               # max wall time HH:MM:SS
#SBATCH --partition=medium            # queue/partition name
#SBATCH --array=1-200%50              # job array: tasks 1-200, max 50 concurrent
```

### Key SLURM commands
```bash
sbatch script.sh          # submit a job
squeue -u $USER           # see your jobs in the queue
scancel <jobid>           # cancel a job
sacct -j <jobid>          # accounting info after job completes
sinfo                     # see partition/node status
```

---

## 6. Python implementation — generate SLURM scripts programmatically

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("NB06: SLURM and HPC Cluster Workflow")
print("No cluster required — this notebook generates and explains SLURM scripts.")

In [ ]:
# --- Generate a SLURM script programmatically ---

def generate_slurm_script(
    job_name: str,
    n_tasks: int,
    cpus_per_task: int,
    mem_gb: int,
    hours: int,
    partition: str,
    command: str,
    max_concurrent: int = 50,
    log_dir: str = 'logs',
) -> str:
    """
    Generate a SLURM batch script for a job array.
    
    The script is returned as a string. Write it to a .sh file and submit with `sbatch`.
    
    Parameters
    ----------
    job_name : str
        Name visible in squeue.
    n_tasks : int
        Number of array tasks (one per sample/file).
    cpus_per_task : int
        Cores per task. Set this to the thread count your tool supports.
    mem_gb : int
        Total memory per task in GB.
    hours : int
        Maximum wall time in hours.
    partition : str
        SLURM partition/queue name.
    command : str
        The command to run for each task. Use $SLURM_ARRAY_TASK_ID to index the sample.
    max_concurrent : int
        Maximum concurrent array tasks (throttle to avoid overloading the cluster).
    log_dir : str
        Directory for log files. Will be created by the script if absent.
    """
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={log_dir}/{job_name}_%A_%a.out
#SBATCH --error={log_dir}/{job_name}_%A_%a.err
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task={cpus_per_task}
#SBATCH --mem={mem_gb}G
#SBATCH --time={hours:02d}:00:00
#SBATCH --partition={partition}
#SBATCH --array=1-{n_tasks}%{max_concurrent}

# ── Environment setup ──────────────────────────────────────────────────────────
set -euo pipefail                          # exit on error, unbound var, pipe failure
module load conda                          # if your cluster uses environment modules
conda activate compbio                     # activate your conda environment

# ── Log task info ──────────────────────────────────────────────────────────────
echo "Job: $SLURM_JOB_ID  Array task: $SLURM_ARRAY_TASK_ID"
echo "Node: $(hostname)  CPUs: $SLURM_CPUS_PER_TASK  Memory: {mem_gb}G"
echo "Started: $(date)"

# ── Main command ───────────────────────────────────────────────────────────────
{command}

echo "Finished: $(date)"
"""
    return script

# Example: HISAT2 alignment for 200 samples
hisat2_command = """\
# Load sample name from an ordered list (one sample per line)
SAMPLE_LIST=samples.txt
SAMPLE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" $SAMPLE_LIST)

# Input/output paths
FASTQ_DIR=/data/fastq
ALIGN_DIR=/data/alignments
GENOME_INDEX=/data/ref/hg38/hisat2_index/genome

mkdir -p $ALIGN_DIR

# Run HISAT2 alignment
hisat2 \\\
  --threads $SLURM_CPUS_PER_TASK \\\
  -x $GENOME_INDEX \\\
  -1 $FASTQ_DIR/${SAMPLE}_R1.fastq.gz \\\
  -2 $FASTQ_DIR/${SAMPLE}_R2.fastq.gz \\\
  --dta \\\
  2> $ALIGN_DIR/${SAMPLE}_hisat2.log \\\
| samtools sort -@ 4 -o $ALIGN_DIR/${SAMPLE}.bam

samtools index $ALIGN_DIR/${SAMPLE}.bam"""

slurm_script = generate_slurm_script(
    job_name='hisat2_align',
    n_tasks=200,
    cpus_per_task=8,
    mem_gb=32,
    hours=3,
    partition='medium',
    command=hisat2_command,
    max_concurrent=50,
)

print(slurm_script)

In [ ]:
# --- Snakemake: minimal RNA-seq pipeline ---

snakefile_content = '''
# Snakefile — minimal RNA-seq pipeline: FASTQ → HISAT2 → HTSeq-count
#
# Run locally:   snakemake --cores 8
# Run on SLURM:  snakemake --executor slurm --default-resources slurm_partition=medium mem_mb=8000

configfile: "config.yaml"   # sample list, paths, tool parameters

SAMPLES = config["samples"]     # e.g. ["SRR123456", "SRR123457", ...]
GENOME  = config["genome_index"]
GTF     = config["gtf"]

# ── Target rule: request the final outputs to trigger the whole DAG ──────────
rule all:
    input:
        expand("results/counts/{sample}.counts", sample=SAMPLES)

# ── Step 1: Quality trimming with Trimmomatic ────────────────────────────────
rule trim:
    input:
        r1 = "data/fastq/{sample}_R1.fastq.gz",
        r2 = "data/fastq/{sample}_R2.fastq.gz",
    output:
        r1 = "results/trimmed/{sample}_R1.fastq.gz",
        r2 = "results/trimmed/{sample}_R2.fastq.gz",
    threads: 4
    resources:
        mem_mb = 4000,
        runtime = 60,                 # minutes
    shell:
        """
        trimmomatic PE -threads {threads} \\\
            {input.r1} {input.r2} \\\
            {output.r1} /dev/null \\\
            {output.r2} /dev/null \\\
            SLIDINGWINDOW:4:20 MINLEN:36
        """

# ── Step 2: Alignment with HISAT2 ───────────────────────────────────────────
rule align:
    input:
        r1 = "results/trimmed/{sample}_R1.fastq.gz",
        r2 = "results/trimmed/{sample}_R2.fastq.gz",
    output:
        bam = "results/alignments/{sample}.bam",
        bai = "results/alignments/{sample}.bam.bai",
    threads: 8
    resources:
        mem_mb = 32000,
        runtime = 120,
    shell:
        """
        hisat2 --threads {threads} -x {GENOME} \\\
            -1 {input.r1} -2 {input.r2} --dta \\\
            | samtools sort -@ 4 -o {output.bam}
        samtools index {output.bam}
        """

# ── Step 3: Count reads per gene with HTSeq-count ───────────────────────────
rule count:
    input:
        bam = "results/alignments/{sample}.bam",
    output:
        counts = "results/counts/{sample}.counts",
    threads: 1
    resources:
        mem_mb = 8000,
        runtime = 90,
    shell:
        """
        htseq-count -f bam -r pos -s no \\\
            {input.bam} {GTF} > {output.counts}
        """
'''

print("Snakefile (minimal 3-step RNA-seq pipeline):")
print(snakefile_content)

print("\nKey concepts:")
print("  rule all:         Target rule — Snakemake works BACKWARDS from outputs to inputs")
print("  expand():         Generates all sample-specific output paths")
print("  threads/resources: Passed to SLURM via --executor slurm")
print("  {wildcards.sample}: Snakemake substitutes the sample name into each path")
print("  Dependency graph: count depends on align depends on trim — enforced by Snakemake")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: HPC cluster architecture
ax = axes[0]
ax.axis('off')
arch_text = (
    "HPC Cluster Architecture\n"
    "\n"
    "  [Login Node]\n"
    "  ├── Submit jobs (sbatch)\n"
    "  ├── Edit scripts\n"
    "  └── Monitor queue (squeue)\n"
    "       |\n"
    "       v SLURM scheduler\n"
    "  [Compute Nodes]\n"
    "  ├── node001  8 cores, 64GB\n"
    "  ├── node002  8 cores, 64GB\n"
    "  ├── ...\n"
    "  └── node128  8 cores, 64GB\n"
    "       |\n"
    "  [Shared Storage]\n"
    "  ├── /data/     (large files)\n"
    "  ├── /home/     (small files)\n"
    "  └── /scratch/  (temp, fast, purged)\n"
    "\n"
    "  NEVER run computation on the\n"
    "  login node — use sbatch!"
)
ax.text(0.05, 0.95, arch_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eaf2f8', alpha=0.8))
ax.set_title('HPC Cluster Architecture', fontweight='bold')

# Panel 2: Job queue states
ax = axes[1]
ax.axis('off')
states_text = (
    "SLURM Job States\n\n"
    "PENDING (PD)\n"
    "  Waiting for resources\n"
    "  Reason: Priority / Resources\n\n"
    "RUNNING (R)\n"
    "  Executing on a compute node\n"
    "  squeue shows node name\n\n"
    "COMPLETING (CG)\n"
    "  Cleaning up after exit\n\n"
    "COMPLETED (CD)\n"
    "  Finished successfully\n\n"
    "FAILED (F)\n"
    "  Non-zero exit code\n"
    "  Check .err log file\n\n"
    "TIMEOUT (TO)\n"
    "  Exceeded --time limit\n"
    "  Increase time or optimize\n\n"
    "OUT_OF_MEMORY (OOM)\n"
    "  Exceeded --mem limit\n"
    "  Increase --mem"
)
ax.text(0.05, 0.98, states_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#fef9e7', alpha=0.8))
ax.set_title('SLURM Job Queue States', fontweight='bold')

# Panel 3: Snakemake DAG
ax = axes[2]
ax.axis('off')
dag_text = (
    "Snakemake DAG\n"
    "(3 samples shown)\n\n"
    "  trim(SRR1) trim(SRR2) trim(SRR3)\n"
    "      |           |          |\n"
    "      v           v          v\n"
    "  align(SRR1) align(SRR2) align(SRR3)\n"
    "      |           |          |\n"
    "      v           v          v\n"
    "  count(SRR1) count(SRR2) count(SRR3)\n"
    "      |           |          |\n"
    "       \\          |         /\n"
    "        \\         v        /\n"
    "             rule all\n\n"
    "Snakemake executes each column\n"
    "in parallel (independent samples)\n"
    "and each row sequentially\n"
    "(trim must finish before align)\n\n"
    "snakemake --dag | dot -Tpng\n"
    "  generates a visual DAG image"
)
ax.text(0.05, 0.98, dag_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eafaf1', alpha=0.8))
ax.set_title('Snakemake Pipeline DAG', fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/nb06_slurm.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **SLURM script for variant calling:** Generate a SLURM job array script for GATK HaplotypeCaller, 50 samples, 4 cores each, 16 GB RAM, 4 hours, max 10 concurrent tasks. Use the `generate_slurm_script` function from this notebook.

2. **Snakemake Wildcard:** Add a rule `merge_counts` to the Snakefile that takes all `results/counts/{sample}.counts` files as input and produces a single merged TSV matrix using a Python script. What Snakemake feature do you use to specify "all samples" as input?

3. **Environment management:** Research: what is the difference between `module load conda` (environment modules) and `conda activate` in a SLURM script? Why can't you just put `conda activate compbio` without sourcing conda first?

4. **Resource estimation:** You have an alignment job that uses 6 GB RAM peak (measured from `sacct --format=MaxRSS`). How much `--mem` should you request? What happens if you request exactly 6 GB and the job slightly exceeds it?

## 12. Reflection

*Write here: What is the key difference between a SLURM job array and parallel processes on a single node? When would you use `--ntasks=8` instead of `--cpus-per-task=8`? What is `$SCRATCH` and why should you use it instead of `$HOME` for large intermediate files?*

---

## Papers referenced

- Mölder, F. et al. (2021). "Sustainable data analysis with Snakemake." *F1000Research* 10:33.

## References

- SLURM documentation: https://slurm.schedmd.com/documentation.html
- Snakemake docs: https://snakemake.readthedocs.io/en/stable/
- Snakemake SLURM executor plugin: https://snakemake.github.io/snakemake-plugin-catalog/plugins/executor/slurm.html
- Harvard FAS SLURM guide: https://docs.rc.fas.harvard.edu/kb/running-jobs/

## Future work / open questions

- Nextflow is an alternative to Snakemake widely used at EMBL/EBI (the nf-core project). What are the key differences?
- Cloud HPC (AWS Batch, Google Life Sciences, Azure Batch) uses the same concepts as SLURM but without a physical cluster. How does the resource request model translate?